# So Sánh FP-Growth (Cổ điển) và LightGCN (Hiện đại)

## Nguồn dữ liệu
Notebook này đọc từ **2 Data Mart** trích xuất từ **Data Warehouse (DuckDB — Snowflake Schema)**:

| Model | Data Mart | Mô tả |
|---|---|---|
| FP-Growth | `mart_basket.csv` | Transaction list (order_id + product_name) |
| LightGCN  | `mart_user_item.csv` | User-Item interaction pairs |

## Luồng dữ liệu đầy đủ
```
data/raws/*.csv (6 files, 32M rows)
    → [Stage 1 - Extract]  : CSV → Parquet (data/staging/)
    → [Stage 2 - Transform]: business logic (null fill, enrichment, mapping)
    → [Stage 3 - Load]     : DuckDB Snowflake Schema (data/warehouse/instacart_dw.duckdb)
    → [Data Mart 1]        → data/warehouse/marts/mart_basket.csv     → FP-Growth
    → [Data Mart 2]        → data/warehouse/marts/mart_user_item.csv  → LightGCN
```

In [ ]:
# ── Imports & Config ──────────────────────────────────────────────────────
import sys, time, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
from eval_utils import create_holdout_test_set, predict_apriori, calculate_hit_rate
from gnn_utils  import prepare_graph_data, train_lightgcn, predict_gnn_bipartite

# ── Đường dẫn Data Mart (Output từ Data Warehouse) ───────────────────────
MART_BASKET    = '../data/warehouse/marts/mart_basket.csv'
MART_USER_ITEM = '../data/warehouse/marts/mart_user_item.csv'

TOP_N_PRODUCTS = 1000    # Giới hạn cho FP-Growth
N_TEST_ORDERS  = 500     # Số đơn dùng để đánh giá

print('Config:')
print(f'  mart_basket    : {MART_BASKET}')
print(f'  mart_user_item : {MART_USER_ITEM}')

In [ ]:
# ── Load Data từ Data Mart ────────────────────────────────────────────────
print('Đọc mart_basket.csv từ Data Warehouse...')
df_basket = pd.read_csv(MART_BASKET)

print(f'  mart_basket  : {len(df_basket):,} dòng | {df_basket["order_id"].nunique():,} orders | {df_basket["product_name"].nunique():,} products')

# Lọc top sản phẩm phổ biến
top_products = df_basket['product_name'].value_counts().nlargest(TOP_N_PRODUCTS).index
df = df_basket[df_basket['product_name'].isin(top_products)].copy()

# Tạo train/test split
train_df, test_cases = create_holdout_test_set(df, n_test_orders=N_TEST_ORDERS, seed=42)
print(f'\nSplit: Train = {train_df["order_id"].nunique():,} orders | Test = {len(test_cases)} orders')

In [ ]:
# ── FP-Growth Training ────────────────────────────────────────────────────
t0 = time.time()

train_transactions = train_df.groupby('order_id')['product_name'].apply(list).tolist()
te = TransactionEncoder()
te_ary = te.fit(train_transactions).transform(train_transactions, sparse=True)
basket_sets = pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)

freq_items = fpgrowth(basket_sets, min_support=0.001, use_colnames=True)
rules = association_rules(freq_items, metric='lift', min_threshold=1.0)
rules = rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])

apriori_time = time.time() - t0
print(f'FP-Growth hoàn tất: {apriori_time:.2f}s — {len(rules):,} luật kết hợp')

In [ ]:
# ── LightGCN Training ─────────────────────────────────────────────────────
# LightGCN dùng mart_user_item (user-item bipartite graph)
# nhưng để so sánh công bằng, train trên cùng train_df

edge_index, metadata = prepare_graph_data(
    train_df,
    all_products=df['product_name'].unique()
)
model, item_emb, gnn_time = train_lightgcn(edge_index, metadata, epochs=30, dim=64)
print(f'LightGCN hoàn tất: {gnn_time:.2f}s')

In [ ]:
# ── Đánh giá: Hit Rate (Precision & Recall) ───────────────────────────────
print('=' * 75)
print('  ĐÁNH GIÁ TRÊN TỔNG SỐ LẦN DỰ ĐOÁN ĐÚNG (TRUE POSITIVES)')
print('  [Nguồn dữ liệu: Data Mart từ Data Warehouse DuckDB]')
print('=' * 75)

# FP-Growth
apriori_pred_fn = lambda inp, k: predict_apriori(inp, rules=rules, k=k)
hits_a, targets_a, prec_a, rec_a = calculate_hit_rate(
    test_cases, 'apriori', apriori_pred_fn, k=10
)
print(f'[FP-Growth]  Đoán trúng: {hits_a:,} / {targets_a:,} món '
      f'(Precision: {prec_a*100:.2f}%  |  Recall: {rec_a*100:.2f}%)')

# LightGCN
gnn_pred_fn = lambda inp, k: predict_gnn_bipartite(inp, item_emb, metadata, k=k)
hits_g, targets_g, prec_g, rec_g = calculate_hit_rate(
    test_cases, 'gnn', gnn_pred_fn, k=10
)
print(f'[LightGCN]   Đoán trúng: {hits_g:,} / {targets_g:,} món '
      f'(Precision: {prec_g*100:.2f}%  |  Recall: {rec_g*100:.2f}%)')

print('=' * 75)

# So sánh
winner_prec   = 'FP-Growth' if prec_a > prec_g else 'LightGCN'
winner_recall = 'FP-Growth' if rec_a  > rec_g  else 'LightGCN'
print(f'\nPrecision tốt hơn : {winner_prec}')
print(f'Recall tốt hơn    : {winner_recall}')
print(f'Train nhanh hơn   : {"FP-Growth" if apriori_time < gnn_time else "LightGCN"} '
      f'({min(apriori_time, gnn_time):.2f}s vs {max(apriori_time, gnn_time):.2f}s)')

In [ ]:
# ── Bảng tổng kết ─────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model'          : ['FP-Growth', 'LightGCN'],
    'Train Time (s)' : [round(apriori_time, 2), round(gnn_time, 2)],
    'Rules / Params' : [len(rules), '2 layers, dim=64'],
    'True Positives' : [hits_a, hits_g],
    'Precision (%)'  : [round(prec_a * 100, 2), round(prec_g * 100, 2)],
    'Recall (%)'     : [round(rec_a  * 100, 2), round(rec_g  * 100, 2)],
    'Data Source'    : ['mart_basket.csv (DW)', 'mart_basket.csv (DW)'],
})

print('\nBẢNG SO SÁNH KẾT QUẢ:')
print(summary.to_string(index=False))